# Student Dropout Prediction using Logistic Regression

This notebook implements a logistic regression classifier to predict student dropout based on cleaned features. 

## 1. Import Required Libraries

In [ ]:
# Import required libraries
import os
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('default')
sns.set_palette("husl")

## 2. Load and Explore the Dataset

In [ ]:
# Load the dataset (adjust path as needed)
current_user = os.getlogin()
PROCESSED_DIR = Path(f"/home/{current_user}/local-share/processed")
df = pd.read_csv(PROCESSED_DIR / 'feature_selection.csv')

## 3. Split Data

In [ ]:
# Split data based on the 'set' column
train_df = df[df['set'] == 'train'].copy()
val_df = df[df['set'] == 'val'].copy()
test_df = df[df['set'] == 'test'].copy()

print(f"Training set shape: {train_df.shape}")
print(f"Validation set shape: {val_df.shape}")
print(f"Test set shape: {test_df.shape}")

## 4. Prepare Features and Target Variable

In [ ]:
# Specify the target variable
target_col = 'sdo5_degree_drop_out'

# Exclude non-feature columns (set, target, any IDs)
exclude_cols = ['set', 'SINH_ID', target_col]

# Get feature columns
feature_cols = [col for col in df.columns if col not in exclude_cols]
print(f"Number of features: {len(feature_cols)}")
print(f"Feature columns: {feature_cols}")

# Prepare training data
X_train = train_df[feature_cols]
y_train = train_df[target_col]

# Prepare validation data
X_val = val_df[feature_cols]
y_val = val_df[target_col]

# Prepare test data
X_test = test_df[feature_cols]
y_test = test_df[target_col]

## 5. Feature Scaling for Logistic Regression

In [ ]:
# Logistic Regression is sensitive to feature scaling
# Scale features using StandardScaler (fit on training data only)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling completed!")
print(f"Training features shape after scaling: {X_train_scaled.shape}")
print(f"Validation features shape after scaling: {X_val_scaled.shape}")
print(f"Test features shape after scaling: {X_test_scaled.shape}")

# Convert back to DataFrames for easier handling (optional)
X_train_scaled = pd.DataFrame(X_train_scaled, columns=feature_cols, index=X_train.index)
X_val_scaled = pd.DataFrame(X_val_scaled, columns=feature_cols, index=X_val.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=feature_cols, index=X_test.index)

print(f"\nFeature scaling statistics (training set):")
print(f"Mean: {X_train_scaled.mean().mean():.6f}")
print(f"Std: {X_train_scaled.std().mean():.6f}")

## 6. Training and Evaluation
Using a stratified 10-Fold cross-validation approach.

In [ ]:
# Check for missing values in scaled data
print(f"Missing values in X_train_scaled: {X_train_scaled.isna().sum().sum()}")
print(f"Missing values in X_val_scaled: {X_val_scaled.isna().sum().sum()}")
print(f"Missing values in X_test_scaled: {X_test_scaled.isna().sum().sum()}")

# If there are missing values, impute them with median
from sklearn.impute import SimpleImputer

if X_train_scaled.isna().any().any():
    print("\nImputing missing values with median strategy...")
    imputer = SimpleImputer(strategy='median')
    X_train_scaled_imputed = pd.DataFrame(
        imputer.fit_transform(X_train_scaled),
        columns=feature_cols,
        index=X_train_scaled.index
    )
    X_val_scaled_imputed = pd.DataFrame(
        imputer.transform(X_val_scaled),
        columns=feature_cols,
        index=X_val_scaled.index
    )
    X_test_scaled_imputed = pd.DataFrame(
        imputer.transform(X_test_scaled),
        columns=feature_cols,
        index=X_test_scaled.index
    )
    print("Missing values imputed successfully!")
else:
    X_train_scaled_imputed = X_train_scaled
    X_val_scaled_imputed = X_val_scaled
    X_test_scaled_imputed = X_test_scaled
    print("No missing values found - proceeding with original data")

# Update the scaled datasets to use imputed versions
X_train_scaled = X_train_scaled_imputed
X_val_scaled = X_val_scaled_imputed
X_test_scaled = X_test_scaled_imputed

# Combine training and validation data for cross-validation
# (Keep test set separate for final evaluation)
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.metrics import make_scorer
import time

X_train_val = pd.concat([X_train_scaled, X_val_scaled], axis=0)
y_train_val = pd.concat([y_train, y_val], axis=0)

print(f"\nCombined training + validation set shape: {X_train_val.shape}")
print(f"Target distribution in combined set:")
print(y_train_val.value_counts(normalize=True))

# Set up stratified 10-fold cross-validation
cv_strategy = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Define scoring metrics for cross-validation
scoring_metrics = {
    'accuracy': 'accuracy',
    'roc_auc': 'roc_auc',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1'
}

# Initialize the logistic regression model
cv_model = LogisticRegression(
    random_state=42,
    class_weight='balanced',
    max_iter=1000
)

print("\nStarting stratified 10-fold cross-validation...")
print("This will train and evaluate 10 different models...")

start_time = time.time()

# Perform cross-validation
cv_results = cross_validate(
    cv_model,
    X_train_val,
    y_train_val,
    cv=cv_strategy,
    scoring=scoring_metrics,
    return_train_score=True,
    n_jobs=-1
)

end_time = time.time()
print(f"Cross-validation completed in {end_time - start_time:.2f} seconds")

## 7. Cross-Validation Results Analysis

In [ ]:
# Analyze and visualize cross-validation results
import matplotlib.pyplot as plt
from scipy import stats

print("=" * 80)
print("10-FOLD CROSS-VALIDATION RESULTS")
print("=" * 80)

# Extract test scores (what we care about for model evaluation)
test_scores = {
    'Accuracy': cv_results['test_accuracy'],
    'ROC AUC': cv_results['test_roc_auc'],
    'Precision': cv_results['test_precision'],
    'Recall': cv_results['test_recall'],
    'F1-Score': cv_results['test_f1']
}

# Calculate statistics for each metric
cv_stats = {}
for metric, scores in test_scores.items():
    cv_stats[metric] = {
        'mean': np.mean(scores),
        'std': np.std(scores),
        'min': np.min(scores),
        'max': np.max(scores),
        'median': np.median(scores)
    }

# Display summary statistics
print(f"\n📊 CROSS-VALIDATION PERFORMANCE SUMMARY:")
print(f"{'Metric':<12} {'Mean':<8} {'Std':<8} {'Min':<8} {'Max':<8} {'Median':<8}")
print("-" * 60)
for metric, stats in cv_stats.items():
    print(f"{metric:<12} {stats['mean']:.3f}   {stats['std']:.3f}   {stats['min']:.3f}   {stats['max']:.3f}   {stats['median']:.3f}")

# Store main CV results for later use
cv_mean_accuracy = cv_stats['Accuracy']['mean']
cv_mean_roc_auc = cv_stats['ROC AUC']['mean']

print(f"\n🎯 KEY METRICS:")
print(f"   • Mean CV Accuracy: {cv_mean_accuracy:.3f} ± {cv_stats['Accuracy']['std']:.3f}")
print(f"   • Mean CV ROC AUC: {cv_mean_roc_auc:.3f} ± {cv_stats['ROC AUC']['std']:.3f}")

print("=" * 80)

## 8. Train Final Model and Validation Analysis

In [ ]:
# Train final model on full training+validation set for feature analysis and final evaluation
print("Training final model on combined training+validation data...")

lr_model = LogisticRegression(
    random_state=42,
    class_weight='balanced',
    max_iter=1000
)

# Fit on the combined train+val set
lr_model.fit(X_train_val, y_train_val)

print(f"Final model trained successfully!")
print(f"Number of features used: {lr_model.n_features_in_}")
print(f"Number of iterations: {lr_model.n_iter_[0]}")
print(f"Solver used: {lr_model.solver}")
print(f"Regularization (C): {lr_model.C}")

# For comparison with CV results, let's create a validation subset prediction
# Using a simple holdout for visualization purposes
from sklearn.model_selection import train_test_split

X_temp, X_val_holdout, y_temp, y_val_holdout = train_test_split(
    X_train_val, y_train_val, test_size=0.2, random_state=42, stratify=y_train_val
)

# Train on temp set, predict on holdout
lr_temp = LogisticRegression(random_state=42, class_weight='balanced', max_iter=1000)
lr_temp.fit(X_temp, y_temp)

y_val_pred = lr_temp.predict(X_val_holdout)
y_val_pred_proba = lr_temp.predict_proba(X_val_holdout)[:, 1]

# Calculate validation metrics for comparison
val_accuracy = accuracy_score(y_val_holdout, y_val_pred)
val_roc_auc = roc_auc_score(y_val_holdout, y_val_pred_proba)

print(f"\n=== Holdout Validation Performance (for comparison) ===")
print(f"Holdout Accuracy: {val_accuracy:.4f} (CV Mean: {cv_mean_accuracy:.4f})")
print(f"Holdout ROC AUC: {val_roc_auc:.4f} (CV Mean: {cv_mean_roc_auc:.4f})")

print(f"\n=== Classification Report (Holdout) ===")
print(classification_report(y_val_holdout, y_val_pred))

# Confusion Matrix for holdout
print(f"\n=== Confusion Matrix (Holdout) ===")
cm = confusion_matrix(y_val_holdout, y_val_pred)
print(cm)

# Visualize confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Confusion Matrix - Holdout Validation Set')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

## 9. ROC Curve Analysis

In [ ]:
# Plot ROC Curve for holdout validation
fpr, tpr, thresholds = roc_curve(y_val_holdout, y_val_pred_proba)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {val_roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random baseline')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Logistic Regression Model (Holdout Validation)')
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.show()

# Add note about CV vs holdout
print(f"Note: This ROC curve is based on a single holdout validation.")
print(f"The cross-validation results provide a more robust estimate:")
print(f"CV Mean ROC AUC: {cv_mean_roc_auc:.4f} ± {cv_stats['ROC AUC']['std']:.4f}")

## 10. Feature Coefficient Analysis

In [ ]:
# Get feature coefficients from the trained model
feature_coefficients = pd.DataFrame({
    'feature': feature_cols,
    'coefficient': lr_model.coef_[0],
    'abs_coefficient': np.abs(lr_model.coef_[0])
}).sort_values('abs_coefficient', ascending=False)

print("Feature Coefficient Rankings (by absolute value):")
print(feature_coefficients)

# Plot feature coefficients
plt.figure(figsize=(12, 8))
top_features = feature_coefficients.head(20)  # Show top 20 features
colors = ['red' if coef < 0 else 'blue' for coef in top_features['coefficient']]
plt.barh(range(len(top_features)), top_features['coefficient'], color=colors, alpha=0.7)
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Coefficient Value')
plt.title('Top 20 Feature Coefficients - Logistic Regression Model')
plt.axvline(x=0, color='black', linestyle='-', alpha=0.3)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# Display top 10 most important features (by absolute coefficient)
print(f"\nTop 10 most important features (by |coefficient|):")
for i, row in feature_coefficients.head(10).iterrows():
    direction = "increases" if row['coefficient'] > 0 else "decreases"
    print(f"{row['feature']}: {row['coefficient']:.4f} ({direction} dropout probability)")

print(f"\nModel intercept: {lr_model.intercept_[0]:.4f}")

## 11. Final Evaluation on Test Set

In [ ]:
# Use the original logistic regression model for final evaluation
final_model = lr_model
print("Using original logistic regression model for final evaluation")

# Make predictions on test set
y_test_pred = final_model.predict(X_test_scaled)
y_test_pred_proba = final_model.predict_proba(X_test_scaled)[:, 1]

# Calculate test metrics
test_accuracy = accuracy_score(y_test, y_test_pred)
test_roc_auc = roc_auc_score(y_test, y_test_pred_proba)

print("\n=== Final Test Set Performance ===")
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test ROC AUC Score: {test_roc_auc:.4f}")

print("\n=== Test Set Classification Report ===")
print(classification_report(y_test, y_test_pred))

# Test set confusion matrix
print("\n=== Test Set Confusion Matrix ===")
test_cm = confusion_matrix(y_test, y_test_pred)
print(test_cm)

# Visualize test confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(test_cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Confusion Matrix - Test Set')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

# Summary of model performance across all sets
print("\n=== Model Performance Summary ===")
print(f"Validation Accuracy: {val_accuracy:.4f}")
print(f"Validation ROC AUC: {val_roc_auc:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test ROC AUC: {test_roc_auc:.4f}")

## 12. Comprehensive Model Evaluation and Analysis

In [ ]:
# Comprehensive Model Evaluation Analysis

print("=" * 80)
print("COMPREHENSIVE MODEL EVALUATION ANALYSIS")
print("=" * 80)

# Calculate baseline accuracy (predicting majority class)
test_dropout_rate = y_test.mean()
baseline_accuracy = max(test_dropout_rate, 1 - test_dropout_rate)

print(f"\n📊 DATASET CHARACTERISTICS:")
print(f"   • Test set size: {len(y_test):,} students")
print(f"   • Dropout rate: {test_dropout_rate:.1%}")
print(f"   • Non-dropout rate: {1-test_dropout_rate:.1%}")
print(f"   • Baseline accuracy (majority class): {baseline_accuracy:.1%}")

print(f"\n🎯 MODEL PERFORMANCE vs BASELINE:")
print(f"   • Test Accuracy: {test_accuracy:.1%} (vs {baseline_accuracy:.1%} baseline)")
print(f"   • Accuracy improvement: {test_accuracy - baseline_accuracy:+.1%}")
print(f"   • ROC AUC: {test_roc_auc:.3f} (0.5 = random, 1.0 = perfect)")

# Performance analysis
if test_accuracy > baseline_accuracy:
    print(f"   ✅ Model beats baseline by {test_accuracy - baseline_accuracy:.1%}")
else:
    print(f"   ❌ Model underperforms baseline by {baseline_accuracy - test_accuracy:.1%}")

# ROC AUC interpretation
if test_roc_auc > 0.8:
    auc_rating = "Excellent"
elif test_roc_auc > 0.7:
    auc_rating = "Good"
elif test_roc_auc > 0.6:
    auc_rating = "Fair"
else:
    auc_rating = "Poor"

print(f"   📈 AUC Rating: {auc_rating}")

# Confusion Matrix Analysis
tn, fp, fn, tp = test_cm.ravel()
sensitivity = tp / (tp + fn)  # True Positive Rate (Recall)
specificity = tn / (tn + fp)  # True Negative Rate
precision = tp / (tp + fp)    # Positive Predictive Value
npv = tn / (tn + fn)         # Negative Predictive Value

print(f"\n🔍 DETAILED PERFORMANCE METRICS:")
print(f"   • Sensitivity (Recall): {sensitivity:.1%} - Correctly identified dropouts")
print(f"   • Specificity: {specificity:.1%} - Correctly identified non-dropouts")
print(f"   • Precision: {precision:.1%} - Accuracy of dropout predictions")
print(f"   • Negative Predictive Value: {npv:.1%} - Accuracy of non-dropout predictions")

print(f"\n🎭 CONFUSION MATRIX BREAKDOWN:")
print(f"   • True Negatives: {tn:,} (Correctly predicted non-dropouts)")
print(f"   • False Positives: {fp:,} (Incorrectly predicted as dropouts)")
print(f"   • False Negatives: {fn:,} (Missed dropouts)")
print(f"   • True Positives: {tp:,} (Correctly predicted dropouts)")

# Business Impact Analysis
missed_dropouts_rate = fn / (tp + fn)
false_alarm_rate = fp / (tn + fp)

print(f"\n💼 BUSINESS IMPACT:")
print(f"   • Missed dropout rate: {missed_dropouts_rate:.1%} ({fn:,} students)")
print(f"   • False alarm rate: {false_alarm_rate:.1%} ({fp:,} students)")

print(f"\n🔧 MODEL CONFIGURATION:")
print(f"   • Model type: Logistic Regression")
print(f"   • Parameters: Default with class_weight='balanced'")
print(f"   • Regularization: C={final_model.C}")
print(f"   • Penalty: {final_model.penalty}")
print(f"   • Solver: {final_model.solver}")
print(f"   • Max iterations: {final_model.max_iter}")

print(f"\n🎯 RECOMMENDATIONS:")
if test_accuracy < baseline_accuracy:
    print("   ❗ Model performance is below baseline - consider:")
    print("     - Feature engineering and selection")
    print("     - Different algorithms (Random Forest, XGBoost, SVM)")
    print("     - More data collection")
    print("     - Different regularization approaches")
elif test_roc_auc < 0.7:
    print("   ⚠️  Model shows limited predictive power - consider:")
    print("     - Feature selection/engineering")
    print("     - Ensemble methods")
    print("     - Non-linear feature transformations")
    print("     - Polynomial features")
    print("     - Hyperparameter tuning (GridSearchCV or RandomizedSearchCV)")
else:
    print("   ✅ Model shows reasonable performance but could be improved:")
    print("     - Ensemble methods for better accuracy")
    print("     - Feature interaction terms")
    print("     - Advanced regularization (ElasticNet)")
    print("     - Threshold optimization for business needs")
    print("     - Hyperparameter tuning for optimal performance")

print(f"\n🔗 LOGISTIC REGRESSION INSIGHTS:")
print("   • Linear model - assumes linear relationship between features and log-odds")
print("   • Coefficients represent log-odds ratios")
print("   • Good baseline for interpretable predictions")
print("   • May struggle with complex non-linear patterns")
print("   • This experiment uses default parameters for quick exploration")

print("\n" + "=" * 80)